In [13]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [14]:
from typing import Tuple
from sklearn.ensemble._forest import _generate_sample_indices


class BaseUncEstRf:
    def __init__(
        self,
        base_model: RandomForestClassifier = RandomForestClassifier(
            criterion="log_loss",  # very important to have accurate probabilities
            min_samples_leaf=20,
            max_samples=None,  # very important _set_train_data_per_tree logic is created based on this assumption
            max_depth=3,
            random_state=42,
        ),
    ):
        self.base_model = base_model

        self.uc_data = {}  # to be defined in fit
        self.model = None  # to be defined in fit

        self.narms = None  # to be defined in fit
        self.idx_arms = None  # to be defined in fit
        self.nfeats = None  # to be defined in fit
        self.x_train = None  # to be defined in fit. Arm column is the last column of this array.
        self.y_train = None  # to be defined in fit

    def fit(self, c_train: np.array, a_train: np.array, y_train: np.array, exp_arms=None):
        """ "Fit Random Forest model and extract training data points per tree.

        Parameters
        ----------
        c_train : np.array of shape (n_samples, n_context_features)
            Contextual features for training.
        a_train : np.array of shape (n_samples, ) or (n_samples, 1)
            Action chosen for each sample in training. Actions are expected to be encoded in the range [0, {total_arms - 1}]
        y_train : np.array of shape (n_samples, )
            Rewards obtained for each sample in training.
        exp_arms : int, optional
            Number of arms to see in vector a_train.
            Implements a security check to ensure training data is consistent with the expected number of arms.
        """

        self.nfeats = c_train.shape[1]
        self.idx_arms = list(np.sort(np.unique(a_train)))
        self.narms = len(self.idx_arms)

        if exp_arms is not None:
            if self.narms != exp_arms:
                raise ValueError(f"Expected {exp_arms} arms, but found {self.narms} unique arms in a_train.")
        if not np.issubdtype(a_train.dtype, np.integer):
            raise ValueError(f"Expected a_train to be of integer type, but found {a_train.dtype}.")
        if self.idx_arms != list(range(self.narms)):
            raise ValueError(
                f"Expected arms to be encoded in the range [0, {self.narms - 1}], but found arms: {self.idx_arms}."
            )

        self.x_train = np.column_stack((c_train, a_train))
        self.y_train = y_train

        self.model = self.base_model.fit(
            self.x_train,
            self.y_train,
        )
        self._set_train_data_per_tree()

    def _set_train_data_per_tree(self):
        """Compute necessary self.uc_data object to do the uncetainty estimation.

        Notes
        ----------
        self.uc_data : dict
            Dictionary where each key is a tree index and the value is another dictionary containing:
            - 'leaf_counts': dict mapping leaf index to the number of samples in that leaf.
            - 'leaf_avg_vals': dict mapping leaf index to the average value of the samples in that leaf.
            - 'leaf_var_vals': dict mapping leaf index to the variance of the values of the samples in that leaf.
            - 'x_samples': np.array of shape (n_samples, n_context_features + 1)
                Contextual features + arm selected for the training samples of the tree.
            - 'y_samples': np.array of shape (n_samples, )
                Rewards for the training samples of the tree.
            - 'leaf_ids_train': np.array of shape (n_samples, )
                Leaf index for each sample in the training set (x_samples) of the tree.
        """
        self.rf_avg = 1 / self.model.n_estimators
        n_samples = len(self.x_train)
        n_samples_bootstrap = n_samples  # This is true if max_samples=None in RandomForestRegressor.
        # Be careful if other RandomForestRegressor parameters of boosting are used, because this can change the logic of extracting the training samples of a given tree.

        train_indices = [
            _generate_sample_indices(self.model.estimators_[sel_tree].random_state, n_samples, n_samples_bootstrap)
            for sel_tree in range(len(self.model.estimators_))
        ]

        x_train_trees = [self.x_train[indices] for indices in train_indices]
        # #####OPTION ABEL 1st attempt:
        # y_train_trees = [self.y_train[indices] for indices in train_indices]
        #####OPTION what it seems researchers do:
        y_train_trees = [self.y_train[indices] * self.rf_avg for indices in train_indices]

        for sel_tree in range(len(self.model.estimators_)):
            self.uc_data[sel_tree] = {}

            leaf_ids = self.model.estimators_[sel_tree].apply(
                x_train_trees[sel_tree]
            )  # This is the leaf index for each sample in the training set of the tree sel_tree

            leaf_tree_idxs, leaf_tree_count = np.unique(leaf_ids, return_counts=True)

            self.uc_data[sel_tree]["leaf_counts"] = dict(zip(leaf_tree_idxs, leaf_tree_count))

            self.uc_data[sel_tree]["leaf_avg_vals"] = dict(
                zip(
                    leaf_tree_idxs,
                    [y_train_trees[sel_tree][leaf_ids == leaf_idx].mean() for leaf_idx in leaf_tree_idxs],
                )
            )

            # #####OPTION ABEL 1st attempt:
            # self.uc_data[sel_tree]['leaf_var_vals'] = dict(
            #     zip(
            #         leaf_tree_idxs,
            #         [y_train_trees[sel_tree][leaf_ids == leaf_idx].var() for leaf_idx in leaf_tree_idxs]
            #         )
            #     )
            ####OPTION what it seems researchers do:
            dic_leaf_tree_count = self.uc_data[sel_tree]["leaf_counts"]
            self.uc_data[sel_tree]["leaf_var_vals"] = dict(
                zip(
                    leaf_tree_idxs,
                    [
                        y_train_trees[sel_tree][leaf_ids == leaf_idx].var() / dic_leaf_tree_count[leaf_idx]
                        for leaf_idx in leaf_tree_idxs
                    ],
                )
            )

            self.uc_data[sel_tree]["x_samples"] = x_train_trees[sel_tree]
            self.uc_data[sel_tree]["y_samples"] = y_train_trees[sel_tree]
            self.uc_data[sel_tree]["leaf_ids_train"] = (
                leaf_ids  # This is the leaf index for each sample in the training set of the tree sel_tree
            )

    def _set_whole_train_data_per_tree(self):
        """Compute necessary self.uc_data object to do the uncetainty estimation. In this case, we use the whole training data, not just the bootstrap samples.

        Notes
        ----------
        self.uc_data : dict
            Dictionary where each key is a tree index and the value is another dictionary containing:
            - 'leaf_counts': dict mapping leaf index to the number of samples in that leaf.
            - 'leaf_avg_vals': dict mapping leaf index to the average value of the samples in that leaf.
            - 'leaf_var_vals': dict mapping leaf index to the variance of the values of the samples in that leaf.
            - 'x_samples': np.array of shape (n_samples, n_context_features + 1)
                Contextual features + arm selected for the training samples of the tree.
            - 'y_samples': np.array of shape (n_samples, )
                Rewards for the training samples of the tree.
            - 'leaf_ids_train': np.array of shape (n_samples, )
                Leaf index for each sample in the training set (x_samples) of the tree.
        """

        self._set_data_per_tree(x_val=self.x_train, y_val=self.y_train)

    def _set_data_per_tree(self, x_val: np.array, y_val: np.array):
        """Compute necessary self.uc_data object to do the uncetainty estimation. In this case, we use the provided x_val, y_val. we do not use the bootstrap samples.

        Notes
        ----------
        self.uc_data : dict
            Dictionary where each key is a tree index and the value is another dictionary containing:
            - 'leaf_counts': dict mapping leaf index to the number of samples in that leaf.
            - 'leaf_avg_vals': dict mapping leaf index to the average value of the samples in that leaf.
            - 'leaf_var_vals': dict mapping leaf index to the variance of the values of the samples in that leaf.
            - 'x_samples': np.array of shape (n_samples, n_context_features + 1)
                Contextual features + arm selected for the training samples of the tree.
            - 'y_samples': np.array of shape (n_samples, )
                Rewards for the training samples of the tree.
            - 'leaf_ids_train': np.array of shape (n_samples, )
                Leaf index for each sample in the training set (x_samples) of the tree.
        """
        # #####OPTION ABEL 1st attempt:
        # y_val = y_val
        #####OPTION what it seems researchers do:
        y_val = y_val * self.rf_avg

        for sel_tree in range(len(self.model.estimators_)):
            self.uc_data[sel_tree] = {}

            leaf_ids = self.model.estimators_[sel_tree].apply(
                x_val
            )  # This is the leaf index for each sample in the training set of the tree sel_tree

            leaf_tree_idxs, leaf_tree_count = np.unique(leaf_ids, return_counts=True)

            self.uc_data[sel_tree]["leaf_counts"] = dict(zip(leaf_tree_idxs, leaf_tree_count))

            self.uc_data[sel_tree]["leaf_avg_vals"] = dict(
                zip(leaf_tree_idxs, [y_val[leaf_ids == leaf_idx].mean() for leaf_idx in leaf_tree_idxs])
            )

            #####OPTION ABEL 1st attempt:
            # self.uc_data[sel_tree]['leaf_var_vals'] = dict(
            #     zip(
            #         leaf_tree_idxs,
            #         [y_val[leaf_ids == leaf_idx].var() for leaf_idx in leaf_tree_idxs]
            #         )
            #     )

            ####OPTION what it seems researchers do:
            dic_leaf_tree_count = self.uc_data[sel_tree]["leaf_counts"]
            self.uc_data[sel_tree]["leaf_var_vals"] = dict(
                zip(
                    leaf_tree_idxs,
                    [y_val[leaf_ids == leaf_idx].var() / dic_leaf_tree_count[leaf_idx] for leaf_idx in leaf_tree_idxs],
                )
            )

            self.uc_data[sel_tree]["x_samples"] = x_val
            self.uc_data[sel_tree]["y_samples"] = y_val
            self.uc_data[sel_tree]["leaf_ids_train"] = (
                leaf_ids  # This is the leaf index for each sample in the training set of the tree sel_tree
            )

    # def compute_accurate_variance(self, x_test:np.array) -> np.array:
    #     """Compute accurate variance for the provided x_val, y_val.

    #     Parameters
    #     ----------
    #     x_val : np.array of shape (n_samples, n_context_features + 1)
    #         Contextual features + arm selected for the validation samples.
    #     y_val : np.array of shape (n_samples, )
    #         Rewards for the validation samples.

    #     Returns
    #     -------
    #     mean_pred : np.array of shape (n_samples, )
    #         Mean prediction for each sample in x_val.
    #     var_pred : np.array of shape (n_samples, )
    #         Variance prediction for each sample in x_val.
    #     """
    #     n_samples = len(x_test)
    #     n_trees = self.model.n_estimators
    #     leaf_indices = np.zeros((n_samples, n_trees))

    #     for sel_tree in range(n_trees):
    #         leaf_indices[:, sel_tree] = self.model.estimators_[sel_tree].apply(x_test)

    #     for sample in range(n_samples):
    #         leaf_per_tree = leaf_indices[sample, :]
    #         leaf_ids = self.uc_data[sel_tree]['leaf_ids_train']
    #         sample_matrix = [self.uc_data[sel_tree]['y_samples'][leaf_ids == leaf_per_tree[sel_tree]] for sel_tree in range(n_trees)]
    #         count_matrix = [self.uc_data[sel_tree]['leaf_counts'][leaf_per_tree[sel_tree]] for sel_tree in range(n_trees)]

    #     return sample_matrix, count_matrix

    def predict_uc(self, x_test: np.array) -> Tuple[np.array, np.array]:
        """ "Compute Mean and Variance prediction of the points contained in x_test where x_test = np.column_stack((c_test, a_test)).

        Parameters & Outputs
        --------------------
        x_test : np.array of shape (n_samples, n_context_features + 1) // note: the "+1" is due to the arm column
            Contextual features + arm selected.
        mean_pred : np.array of shape (n_samples, )
            Mean prediction for each sample in x_test.
        var_pred : np.array of shape (n_samples, )
            Variance prediction for each sample in x_test.
        """
        if self.model is None:
            raise ValueError("Model has not been fitted yet.")

        n_samples = len(x_test)
        n_trees = self.model.n_estimators
        tree_preds = np.zeros((n_samples, n_trees))
        tree_var_preds = np.zeros((n_samples, n_trees))

        for sel_tree in range(n_trees):
            leaf_indices = self.model.estimators_[sel_tree].apply(x_test)
            leaf_vals = self.uc_data[sel_tree]["leaf_avg_vals"]
            tree_preds[:, sel_tree] = [leaf_vals[leaf_idx] for leaf_idx in leaf_indices]

            leaf_var_vals = self.uc_data[sel_tree]["leaf_var_vals"]
            tree_var_preds[:, sel_tree] = [leaf_var_vals[leaf_idx] for leaf_idx in leaf_indices]

            # ############## delete test
            # leaf_vals = self.uc_data[sel_tree]['leaf_avg_vals_test']
            # tree_preds[:, sel_tree] = [leaf_vals[leaf_idx] for leaf_idx in leaf_indices]

        # #####OPTION ABEL 1st attempt:
        # mean_pred = np.mean(tree_preds, axis=1)
        # var_pred = 1/(n_trees**2)*np.sum(tree_var_preds, axis=1)
        # #####OPTION what it seems researchers do:
        mean_pred = np.sum(tree_preds, axis=1)
        var_pred = np.sum(tree_var_preds, axis=1)

        return mean_pred, var_pred

    def predict_arms_uc(self, c_test: np.array) -> Tuple[np.array, np.array]:
        """Compute Mean and Variance prediction for all the arms given the context points in c_test.

        Parameters & Outputs
        --------------------
        c_test : np.array of shape (n_samples, n_context_features)
            Contextual features for training.
        arm_means : np.array of shape (n_samples, self.narms)
            Mean prediction for each sample and arm in c_test.
        arm_vars : np.array of shape (n_samples, self.narms)
            Variance prediction for each sample and arm in c_test.
        """
        arm_means = np.zeros((c_test.shape[0], self.narms))
        arm_vars = np.zeros((c_test.shape[0], self.narms))
        for arm in self.idx_arms:
            a_test = arm * np.ones(c_test.shape[0])
            x_test = np.column_stack((c_test, a_test))
            arm_means[:, arm], arm_vars[:, arm] = self.predict_uc(x_test)

        return arm_means, arm_vars

In [15]:
########## Load dataset 1
RANDOM_STATE = 123
n_features = 5
# Generate a binary classification dataset.
x, y = make_classification(
    n_samples=5000,
    n_features=n_features,
    n_clusters_per_class=1,
    n_informative=5,
    n_redundant=0,
    n_repeated=0,
    random_state=RANDOM_STATE,
)

# Use last column as arm column. Convert values to integers in the range [0, n_features-1].
q1 = pd.DataFrame(x).describe().loc['25%', n_features-1]
q2 = pd.DataFrame(x).describe().loc['50%', n_features-1]
q3 = pd.DataFrame(x).describe().loc['75%', n_features-1]

# print(f"Q1: {q1}, Q2: {q2}, Q3: {q3}")

x[:, -1] = np.where(
    x[:, -1] < q1,
    0,
    np.where(
        x[:, -1] < q2,
        1,
        np.where(
            x[:, -1] < q3,
            2,
            3)
        )
    )

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=20,  # important for the assumptions of sample mean and sample standard deviation in leaf nodes for TEUCB (Tree Ensemble UCB) and TETS (Tree Ensemble Thompson Sampling)
    max_depth=3,
    criterion="log_loss",
    max_samples=None,
    # n_jobs=N_CORES,
    random_state=42,
)

In [17]:
base_uc_rf = BaseUncEstRf(base_model=rf_model)

In [18]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [19]:
c_train = x_train[:, :-1]  # Contextual features
a_train = x_train[:, -1].astype(int)  # Arm selected

c_test = x_test[:, :-1]  # Contextual features
a_test = x_test[:, -1].astype(int)  # Arm selected

In [20]:
base_uc_rf.fit(c_train=c_train, a_train=a_train, y_train=y_train, exp_arms=len(np.unique(a_train)))

In [21]:
rf_preds = base_uc_rf.model.predict_proba(x_test)[
    :, 1
]  # Assuming binary classification, we take the probability of the positive class
means, vars = base_uc_rf.predict_uc(x_test)


print(f"Are both predictions equal? sklearn vs manual implementation {np.allclose(rf_preds, means)}")

Are both predictions equal? sklearn vs manual implementation True


In [25]:
import pandas as pd
results_df = pd.DataFrame({'rf_preds':rf_preds, 'means':means, 'vars':vars, 'std':vars**0.5})
results_df


,rf_preds,means,vars,std
0,0.077779,0.077779,5.212679e-07,0.000722
1,0.085992,0.085992,7.625575e-07,0.000873
2,0.927136,0.927136,1.070896e-06,0.001035
3,0.245531,0.245531,2.076986e-06,0.001441
4,0.949905,0.949905,5.040008e-07,0.000710
...,...,...,...,...
995,0.738597,0.738597,2.635653e-06,0.001623
996,0.589749,0.589749,3.558469e-06,0.001886
997,0.851753,0.851753,1.385850e-06,0.001177
998,0.707148,0.707148,2.016308e-06,0.001420
